# C7-cnn-transfer — Practice p14 — Solution

**(a) Convolution claim.** The reduce convolution has
$m(4m)1^2=4m^2$ weights, the spatial convolution has
$m\cdot m\cdot3^2=9m^2$, and the expand convolution has
$(4m)m1^2=4m^2$. Their sum is exactly $17m^2$.

**(b) BatchNorm claim.** The three BatchNorms serve $m$, $m$, and
$4m$ channels. At two learned scalars per channel they add
$2(m+m+4m)=12m$, so $P(m)=17m^2+12m$.

**(c) Comparison claim.** A full-width $3\times3$ convolution has
$(4m)(4m)3^2=144m^2$ weights. Thus all three bottleneck convolutions
cost $17m^2/(144m^2)=17/144$ of that one layer, independent of $m$.
The saving comes from doing the expensive spatial convolution in the thin
$m$-channel middle rather than at width $4m$.


In [ ]:
# Cache pin (course convention, plan 009): pretrained weights live in the repo's
# gitignored reference/cache/ -- resolve it from the repo root BEFORE importing torch.
import os, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists())
os.environ["TORCH_HOME"] = str(_root / "reference" / "cache" / "torch")

import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

# float32 register (course exception): pretrained resnet50 is a float32 artifact.
# No float64 default here; inputs are cast .to(torch.float32) at the model
# boundary; repeat float32 forwards are bit-identical.
SEED = 20260804

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval()
assert next(model.parameters()).dtype == torch.float32

# P(512) = 17 * 512^2 + 12 * 512 = 4,456,448 + 6,144.
hand_total = 4_462_592
torch_total = sum(p.numel() for p in model.layer4[1].parameters())
anchor_gap = abs(hand_total - torch_total)


### Answer check

In [ ]:
assert hand_total == 17 * 512**2 + 12 * 512
assert torch_total == 4_462_592
assert anchor_gap == 0
